# 04 - Train the purchase-conversion baseline model

Gate 7: label definition, leakage checks, time-based split, baseline logistic
regression, metrics, and saved artifacts.

Reusable logic lives in `src/analytics_ml/training/train.py`; this notebook
orchestrates and displays it. Artifacts are written to `models/` (Git-ignored);
the metrics summary is tracked as `reports/model_baseline.md`.

In [ ]:
import json
from pathlib import Path

import duckdb
import joblib
import pandas as pd
from sklearn.metrics import ConfusionMatrixDisplay

from analytics_ml.training import train as training

pd.set_option("display.width", 160)
print("duckdb:", duckdb.__version__)
print("catalog:", training.CATALOG_PATH.relative_to(training.PROJECT_ROOT).as_posix())

## 1. Leakage control

In [ ]:
training._assert_no_leakage(training.FEATURES)
print("features:", training.FEATURES)
print("forbidden (must never be features):", sorted(training.FORBIDDEN_FEATURES))
print("leakage check: PASS")

## 2. Load the mart

In [ ]:
frame = training.load_dataset()
print("rows:", len(frame))
print("columns:", list(frame.columns))
print("conversion rate:", frame[training.TARGET].mean())
frame.head(3)

## 3. Time-based split

In [ ]:
train_df, test_df = training.split_dataset(frame)
print("train rows:", len(train_df), "positives:", int(train_df[training.TARGET].sum()))
print("test rows:", len(test_df), "positives:", int(test_df[training.TARGET].sum()))
print("train prevalence:", train_df[training.TARGET].mean())
print("test prevalence:", test_df[training.TARGET].mean())

## 4. Train the baseline

In [ ]:
metrics = training.train()
metrics

## 5. Evaluate on the holdout

In [ ]:
pipeline = joblib.load(training.MODELS_DIR / "baseline_logreg.joblib")
holdout_metrics = training.evaluate(pipeline, test_df)
holdout_metrics

## 6. Confusion matrix

In [ ]:
cm = metrics["confusion_matrix"]
print("threshold:", metrics["threshold"])
print("TN, FP:", cm[0])
print("FN, TP:", cm[1])

## 7. Artifacts

In [ ]:
for path in sorted(training.MODELS_DIR.iterdir()):
    print(path.name, path.stat().st_size, "bytes")
print("report:", training.REPORT_PATH.relative_to(training.PROJECT_ROOT).as_posix())

## 8. Verdict

In [ ]:
print("PR-AUC:", metrics["pr_auc"])
print("ROC-AUC:", metrics["roc_auc"])
print("precision:", metrics["precision"])
print("recall:", metrics["recall"])
print("f1:", metrics["f1"])
assert metrics["pr_auc"] > 0.0
assert (training.MODELS_DIR / "baseline_logreg.joblib").exists()
assert training.REPORT_PATH.exists()
print("PASS")